# Frame Extractor


In [1]:
from pathlib import Path
import pandas as pd

# Encuentra la raíz del proyecto tanto si Jupyter se abrió desde aquí como desde notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'bronze' / 'metadata').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

metadata_path = PROJECT_ROOT / 'data' / 'bronze' / 'metadata' / 'plank_metadata.csv'
metadata = pd.read_csv(metadata_path)
metadata

,video_id,Participant_id,excercise,label,camera_view,duration_in_seconds,fps
0,P01,P1,plank,correct,side,2.633,30.0
1,P02,P2,plank,correct,side,5.567,30.0
2,P03,P3,plank,correct,back-side,1.733,30.0
3,P04,P5,plank,correct,side,3.000,30.0
4,P05,P6,plank,correct,side,2.733,30.0
5,P06,P6,plank,correct,side,3.767,30.0
6,P07,P7,plank,correct,side,3.033,30.0
7,P08,P1,plank,incorrect,side,4.567,30.0
8,P09,P2,plank,incorrect,side,1.867,30.0
9,P10,P4,plank,incorrect,side,1.067,30.0


In [2]:
metadata['duration_in_seconds'].describe()

count    14.000000
mean      3.059571
std       1.208381
min       1.067000
25%       2.308250
50%       2.883500
75%       3.767000
max       5.567000
Name: duration_in_seconds, dtype: float64

In [18]:
metadata['total_frames'] = (metadata['fps'] * metadata['duration_in_seconds']).round()
resumen = metadata.groupby('label')['total_frames'].sum().astype(int)
resumen

label
correct      674
incorrect    611
Name: total_frames, dtype: int64

## Extraccion de todos los frames

En esta etapa no se recorta la duración del video: se decodifica de principio a fin y se guarda **cada frame** como una imagen JPEG. La salida queda organizada por `video_id` en `data/silver/extracted_frames/plank/<video_id>/`, por ejemplo `P01/frame_000001.jpg`.

El CSV de metadata no contiene el nombre de archivo original. Por ello, los videos se ordenan con la misma regla determinista que se usó al crear ese CSV (carpeta `correct` antes de `incorrect`, después por nombre). Antes de extraer, se comparan los datos del nombre y de la carpeta contra cada fila para evitar asignar frames al video equivocado. Si los videos o la metadata cambiaron desde entonces, la notebook se detendrá y pedirá regenerar/validar el CSV.

In [ ]:
import math
import re

import cv2
import pandas as pd

# Rutas de entrada/salida, relativas a la raíz del repositorio.
VIDEO_ROOT = PROJECT_ROOT / 'data' / 'bronze' / 'videos' / 'plank'
FRAMES_ROOT = PROJECT_ROOT / 'data' / 'silver' / 'extracted_frames' / 'plank'

if not VIDEO_ROOT.is_dir():
    raise FileNotFoundError(f'No se encontró la carpeta de videos: {VIDEO_ROOT}')
if not metadata_path.is_file():
    raise FileNotFoundError(f'No se encontró el CSV de metadata: {metadata_path}')

# Acepta nombres como good_plank_03_p3_back-side y bad_plank_01_p1_side.
VIDEO_NAME_PATTERN = re.compile(
    r'^(?P<quality>good|bad)_(?P<exercise>[^_]+)_(?P<number>\d+)_'
    r'(?:(?P<participant>p\d+)_)?(?P<camera_view>.+)$',
    re.IGNORECASE,
)
VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.mpeg', '.mpg'}
EXPECTED_COLUMNS = [
    'video_id', 'Participant_id', 'excercise', 'label',
    'camera_view', 'duration_in_seconds', 'fps'
]

metadata = pd.read_csv(metadata_path)
missing_columns = set(EXPECTED_COLUMNS) - set(metadata.columns)
if missing_columns:
    raise ValueError(f'Faltan columnas en {metadata_path}: {sorted(missing_columns)}')

# La ordenación replica la asignación de video_id de metadata_creation.ipynb.
video_paths = []
for label_folder in ('correct', 'incorrect'):
    folder_path = VIDEO_ROOT / label_folder
    if not folder_path.is_dir():
        raise FileNotFoundError(f'No se encontró la carpeta: {folder_path}')
    video_paths.extend(
        path for path in folder_path.rglob('*')
        if path.is_file() and path.suffix.lower() in VIDEO_EXTENSIONS
    )
video_paths.sort(key=lambda path: (path.parent.name.lower(), path.name.lower()))

if len(video_paths) != len(metadata):
    raise ValueError(
        f'Hay {len(video_paths)} videos, pero {len(metadata)} filas en el CSV. '
        'Actualiza la metadata antes de extraer los frames.'
    )

quality_to_label = {'good': 'correct', 'bad': 'incorrect'}
for video_path, row in zip(video_paths, metadata.itertuples(index=False)):
    match = VIDEO_NAME_PATTERN.fullmatch(video_path.stem)
    if match is None:
        raise ValueError(f'Nombre de video no reconocido: {video_path.name}')

    participant_id = match.group('participant')
    participant_id = participant_id.upper() if participant_id else ''
    folder_label = video_path.parent.name.lower()
    filename_label = quality_to_label[match.group('quality').lower()]
    actual_participant = '' if pd.isna(row.Participant_id) else str(row.Participant_id)
    actual_values = (
        actual_participant,
        str(row.excercise).lower(),
        str(row.label).lower(),
        str(row.camera_view),
    )
    expected_values = (
        participant_id,
        match.group('exercise').lower(),
        folder_label,
        match.group('camera_view'),
    )
    if folder_label != filename_label or actual_values != expected_values:
        raise ValueError(
            f'La fila {row.video_id} no corresponde con {video_path.name}. '
            f'CSV={actual_values}; video={expected_values}'
        )

print(f'Metadata y videos validados: {len(video_paths)}')
print(f'Las imágenes se guardarán en: {FRAMES_ROOT}')

### Muestrear a 7 frames por segundo y guardar

Se decodifica cada video de principio a fin, pero solo se guarda un frame por cada instante objetivo separado por `1/7` de segundo (máximo 7 FPS; si el video original tiene menos de 7 FPS, se guardan todos sus frames). Los JPEG anteriores generados en esta carpeta se eliminan antes de comenzar, para que no queden imágenes de la extracción sin muestreo. El primer frame guardado se numera como `000001`. Al final se valida el número esperado de muestras.

In [ ]:
# Frecuencia objetivo de muestreo, en frames por segundo.
TARGET_FPS = 7.0

# Elimina únicamente los JPEG de frames de una ejecución anterior.
# Así no quedan frames antiguos si se vuelve a ejecutar con otro muestreo.
FRAMES_ROOT.mkdir(parents=True, exist_ok=True)
old_frames = list(FRAMES_ROOT.rglob('frame_*.jpg'))
for old_frame in old_frames:
    old_frame.unlink()
print(f'Frames antiguos eliminados: {len(old_frames)}')

extraction_summary = []

for video_path, row in zip(video_paths, metadata.itertuples(index=False)):
    video_id = str(row.video_id)
    if not re.fullmatch(r'P\d{2,}', video_id):
        raise ValueError(f'video_id con formato inesperado en metadata: {video_id}')

    # Cada video tiene su propia carpeta para conservar la correspondencia con metadata.
    output_dir = FRAMES_ROOT / video_id
    output_dir.mkdir(parents=True, exist_ok=True)

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        capture.release()
        raise OSError(f'OpenCV no pudo abrir el video: {video_path}')

    reported_frames = capture.get(cv2.CAP_PROP_FRAME_COUNT)
    source_fps = capture.get(cv2.CAP_PROP_FPS)
    if not math.isfinite(reported_frames) or reported_frames <= 0:
        capture.release()
        raise ValueError(f'El video no reporta un número válido de frames: {video_path}')
    if not math.isfinite(source_fps) or source_fps <= 0:
        capture.release()
        raise ValueError(f'El video no reporta un FPS válido: {video_path}')

    saved_frames = 0
    source_frame_index = 0
    next_sample_frame = 0.0
    source_frame_interval = source_fps / TARGET_FPS
    try:
        while True:
            success, frame = capture.read()
            if not success:
                break

            # El índice del frame se compara con el próximo instante de muestreo.
            if source_frame_index + 1e-9 >= next_sample_frame:
                saved_frames += 1
                frame_path = output_dir / f'frame_{saved_frames:06d}.jpg'
                if not cv2.imwrite(str(frame_path), frame):
                    raise OSError(f'No se pudo guardar el frame: {frame_path}')
                next_sample_frame += source_frame_interval
            source_frame_index += 1
    finally:
        capture.release()

    total_source_frames = int(round(reported_frames))
    # Para videos con menos de 7 FPS se guardan todos los frames disponibles.
    effective_interval = max(source_frame_interval, 1.0)
    expected_frames = min(
        total_source_frames,
        math.floor((total_source_frames - 1) / effective_interval) + 1,
    )
    if saved_frames != expected_frames:
        raise RuntimeError(
            f'Extracción incompleta para {video_path.name}: '
            f'{saved_frames} muestras guardadas; se esperaban {expected_frames} '
            f'a {TARGET_FPS:g} FPS (video original: {source_fps:g} FPS).'
        )

    extraction_summary.append(
        {
            'video_id': video_id,
            'video': video_path.name,
            'source_fps': round(source_fps, 3),
            'target_fps': TARGET_FPS,
            'frames_saved': saved_frames,
        }
    )

summary = pd.DataFrame(extraction_summary)
print(f'Videos procesados: {len(summary)}')
print(f'Total de frames guardados a un máximo de {TARGET_FPS:g} FPS: '
      f'{int(summary["frames_saved"].sum())}')
summary

Al terminar, la estructura tendrá una carpeta por `video_id`, con imágenes numeradas desde `frame_000001.jpg`. Los videos fuente están a 30 FPS, así que cada carpeta tendrá aproximadamente 7 imágenes por segundo de video.